# Semana 3 — O Motor do Aprendizado: Calculus e Gradient Descent
**Fase 1: Math Foundations e ML Core** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/01-Phase-1-Math-and-ML-Core/03-calculus-gradient/notebooks/semana_03_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-03-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/01-Phase-1-Math-and-ML-Core/03-calculus-gradient/docs/Semana-03-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_03_gabarito.ipynb`.

## Passo 1 — Dados: uma reta com ruído

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(42)
x = rng.uniform(-3, 3, 200)
y = 2 * x + 1 + rng.normal(0, 0.5, 200)       # verdade: w=2, b=1
plt.scatter(x, y, s=8); plt.title("y = 2x + 1 + ruído"); plt.show()

## Passo 2 — Previsão e loss

In [ ]:
def prever(x, w, b):
    # [SEU CÓDIGO AQUI]

def mse(y_hat, y):
    # [SEU CÓDIGO AQUI] média do erro ao quadrado

print("Perda com w=0, b=0:", round(mse(prever(x, 0.0, 0.0), y), 3))

## Passo 3 — Gradiente analítico
Implemente as duas partial derivatives do MSE (fórmulas na leitura de conceito).

In [ ]:
def gradientes(x, y, w, b):
    # [SEU CÓDIGO AQUI] devolva (dw, db)

## Passo 4 — Confira com diferenças finitas
Se o gradiente analítico estiver certo, ele bate com a aproximação numérica.

In [ ]:
def gradiente_numerico(x, y, w, b, eps=1e-5):
    # [SEU CÓDIGO AQUI] use (L(θ+eps) - L(θ-eps)) / (2*eps) para w e para b

ga, gn = gradientes(x, y, 0.3, -0.7), gradiente_numerico(x, y, 0.3, -0.7)
print("analítico:", ga, "\nnumérico :", gn)
assert np.allclose(ga, gn, atol=1e-5)

## Passo 5 — O loop de treinamento
Para cada epoch: calcule os gradientes, atualize `w` e `b` na direção oposta, registre a loss.

In [ ]:
def treinar(x, y, lr=0.05, epocas=100, w=None, b=None):
    w = rng.normal() if w is None else w
    b = rng.normal() if b is None else b
    hist = []
    for _ in range(epocas):
        # [SEU CÓDIGO AQUI] gradientes e atualização de w e b
        hist.append((w, b, mse(prever(x, w, b), y)))
    return w, b, hist

w, b, hist = treinar(x, y)
print(f"w={w:.3f} (alvo 2) | b={b:.3f} (alvo 1) | perda final={hist[-1][2]:.4f}")
plt.plot([h[2] for h in hist]); plt.yscale("log"); plt.xlabel("época"); plt.ylabel("MSE"); plt.show()

## Passo 6 — O efeito da learning rate
Treine com `lr` em `[0.001, 0.01, 0.1, 0.3, 0.36]` partindo sempre de `w=0, b=0` e desenhe as curvas no mesmo gráfico.

In [ ]:
for lr in [0.001, 0.01, 0.1, 0.3, 0.36]:
    # [SEU CÓDIGO AQUI] treine e guarde o histórico em h
    plt.plot([p[2] for p in h], label=f"lr={lr}")
plt.yscale("log"); plt.legend(); plt.xlabel("época"); plt.ylabel("MSE"); plt.show()

Qual curva é lenta, qual é boa, qual oscila e qual diverge?

## Passo 7 — Mini-batch SGD
Em vez de usar todos os dados em cada passo, use batches de 16 exemplos embaralhados. A curva fica ruidosa, mas cada passo é barato: é assim que LLMs são treinados.

In [ ]:
def treinar_sgd(x, y, lr=0.05, epocas=20, lote=16):
    w, b, hist = 0.0, 0.0, []
    for _ in range(epocas):
        perm = rng.permutation(len(x))
        for i in range(0, len(x), lote):
            # [SEU CÓDIGO AQUI] selecione o batch com perm, calcule gradientes e atualize
            hist.append(mse(prever(x, w, b), y))
    return w, b, hist

w, b, h = treinar_sgd(x, y)
print(f"w={w:.3f} b={b:.3f}"); plt.plot(h); plt.yscale("log"); plt.xlabel("passo"); plt.show()

## Passo 8 — Por que padronizar: o vale estreito
Multiplique `x` por 100 (como uma coluna em outra unidade) e tente treinar com a mesma taxa.

In [ ]:
x_grande = x * 100
for nome, xs, lr in [("x original, lr=0.05", x, 0.05), ("x*100, lr=0.05", x_grande, 0.05), ("x*100, lr=1e-5", x_grande, 1e-5)]:
    with np.errstate(all="ignore"):
        _, _, h = treinar(xs, y, lr=lr, epocas=50, w=0.0, b=0.0)
    print(f"{nome:22s} -> perda final = {h[-1][2]:.4g}")
# [SEU CÓDIGO AQUI] padronize x_grande (média 0, desvio 1) em x_pad
_, _, h = treinar(x_pad, y, lr=0.05, epocas=50, w=0.0, b=0.0)
print(f"{'x padronizado, lr=0.05':22s} -> perda final = {h[-1][2]:.4g}")

## Passo 9 — Chain rule em uma rede de 1 neurônio oculto
Modelo: `h = tanh(w1·x + b1)`, `ŷ = w2·h + b2`, `L = média((ŷ − y)²)`. Faça o backward na mão, de trás para frente.

In [ ]:
def forward_backward(x, y, p):
    w1, b1, w2, b2 = p
    z = w1 * x + b1; h = np.tanh(z); y_hat = w2 * h + b2
    L = np.mean((y_hat - y) ** 2)
    # [SEU CÓDIGO AQUI] calcule dw1, db1, dw2, db2 aplicando a chain rule
    return L, np.array([dw1, db1, dw2, db2])

p = np.array([0.5, -0.2, 1.3, 0.1]); alvo = np.sin(x)
L, g = forward_backward(x, alvo, p)
g_num = np.array([(forward_backward(x, alvo, p + e)[0] - forward_backward(x, alvo, p - e)[0]) / 2e-5 for e in np.eye(4) * 1e-5])
print("analítico:", g.round(5), "\nnumérico :", g_num.round(5)); assert np.allclose(g, g_num, atol=1e-5)

Você acabou de implementar backpropagation. Na Semana 4 o PyTorch fará exatamente esse cálculo por você.